## Importando e compreendendo os dados como vieram

In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
#padronizando caminhos

BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "datasets" / "dados_recebidos"

produtos_path = DATA_RAW / "produtos_raw.csv"
vendas_path = DATA_RAW / "vendas_2023_2024.csv"
clientes_path = DATA_RAW / "clientes_crm.json"
custos_path = DATA_RAW / "custos_importacao.json"

df_produtos = pd.read_csv(produtos_path)
df_vendas = pd.read_csv(vendas_path)

with open(clientes_path, "r", encoding="utf-8") as f:
    clientes_data = json.load(f)
df_clientes = pd.DataFrame(clientes_data)

with open(custos_path, "r", encoding="utf-8") as f:
    custos_data = json.load(f)
df_custos = pd.DataFrame(custos_data)

## Primeira olhada nas bases

Conferindo o tamanho de cada base e olhando as primeiras linhas para entender melhor a estrutura dos dados.

Aqui a ideia é identificar rapidamente se as colunas fazem sentido, se existe algum problema de formatação e se já aparece alguma inconsistência logo no começo.

In [4]:
print("Produtos:", df_produtos.shape)
print("Vendas:", df_vendas.shape)
print("Clientes:", df_clientes.shape)
print("Custos:", df_custos.shape)

display(df_produtos.head())
display(df_vendas.head())
display(df_clientes.head())
display(df_custos.head())

df_produtos.info()
df_vendas.info()
df_clientes.info()
df_custos.info()

Produtos: (157, 4)
Vendas: (9895, 6)
Clientes: (49, 4)
Custos: (150, 4)


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             157 non-null    str  
 1   price            157 non-null    str  
 2   code             157 non-null    int64
 3   actual_category  157 non-null    str  
dtypes: int64(1), str(3)
memory usage: 5.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 464.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column     Non-Nul

## Problemas na base de produtos

Logo de cara deu pra ver que a coluna de categoria está bem bagunçada.

Tem a mesma categoria escrita de várias formas diferentes, por exemplo:
- ELETRONICOS
- E L E T R Ô N I C O S
- Eletrunicos
- Eletronicz

Isso pode dar problema quando for agrupar ou analisar os dados depois.

Outro ponto é que o preço está como texto, então do jeito que está não dá pra fazer cálculo. Vou precisar converter isso pra número.

## Problemas na base de vendas

A base está mais organizada, mas a coluna de data não segue um padrão único.

Tem datas no formato:
- 2023-09-10  
- 15-09-2024  

Isso pode dar problema na hora de analisar por período, então vou padronizar depois.

## Problemas na base de clientes

Aqui já aparecem alguns problemas mais claros.

Os e-mails não estão todos corretos. Em alguns casos aparece "#" no lugar de "@", o que invalida o contato.

A localização também está bem inconsistente. Cada registro usa um formato diferente, como:
- PE , Recife  
- Rio Grande,RS  
- PA - Santarém Novo  

Do jeito que está, fica difícil usar essa informação pra qualquer análise por região.

Vou precisar organizar isso melhor.

## Problemas na base de custos

Essa base tem um campo com uma estrutura mais complexa (uma lista com histórico de dados).

Do jeito que está, não é tão simples de usar direto. Provavelmente vou precisar tratar isso depois pra facilitar as análises.

In [5]:
df_produtos["actual_category"].value_counts()

actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
Eletronicoz              7
eletrônicos              7
ELETRONICOS              6
E L E T R Ô N I C O S    6
PROPULSAO                6
P R O P U L S Ã O        6
propulsão                6
Eletrunicos              5
eLeTrÔnIcOs              5
Propulção                5
Prop                     5
Propulssão               5
Encoragem                5
Ancorajm                 5
A N C O R A G E M        5
aNcOrAgEm                5
Eletrônicos              4
propulsao                4
eletronicos              3
EletrônicoS              3
Propulçao                3
Ancoragem                3
Ancorajem                3
Eletroniscos             2
Eletronicos              2
pRoPuLsÃo                2
Propulsam                2
AnCoRaGeM                2
ancoragem                2
Ancorajen                2
ELEtRÔNICOS              1
PrOpUlSãO                1
ANCORAGEM                1
Encoragi    

In [7]:
#Padronizando textos

import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    
    texto = texto.upper()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    texto = texto.replace(" ", "")
    
    return texto

df_produtos["categoria_norm"] = df_produtos["actual_category"].apply(normalizar_texto)

def corrigir_categoria(cat):
    if "ELETRONIC" in cat:
        return "ELETRONICOS"
    elif "PROPUL" in cat:
        return "PROPULSAO"
    elif "ANCOR" in cat:
        return "ANCORAGEM"
    else:
        return cat

df_produtos["categoria_final"] = df_produtos["categoria_norm"].apply(corrigir_categoria)

In [8]:
df_produtos["categoria_final"].value_counts()

categoria_final
ANCORAGEM       47
ELETRONICOS     44
PROPULSAO       40
PROPUCAO         8
ELETRUNICOS      5
PROP             5
ENCORAGEM        5
ELETRONISCOS     2
ENCORAGI         1
Name: count, dtype: int64

## Refinando a padronização das categorias

Depois da primeira limpeza, ainda sobraram algumas variações com erro de digitação ou abreviação.

Então aqui vou fazer um segundo ajuste, agora mais direcionado, para consolidar tudo nas três categorias corretas.

In [9]:
def corrigir_categoria_final(cat):
    if cat in ["ELETRONICOS", "ELETRUNICOS", "ELETRONISCOS"]:
        return "ELETRONICOS"
    elif cat in ["PROPULSAO", "PROPUCAO", "PROP"]:
        return "PROPULSAO"
    elif cat in ["ANCORAGEM", "ENCORAGEM", "ENCORAGI"]:
        return "ANCORAGEM"
    else:
        return cat

df_produtos["categoria_final"] = df_produtos["categoria_final"].apply(corrigir_categoria_final)

df_produtos["categoria_final"].value_counts()

categoria_final
PROPULSAO      53
ANCORAGEM      53
ELETRONICOS    51
Name: count, dtype: int64

In [ ]:
#Conferência

df_produtos["actual_category"] = df_produtos["categoria_final"]
df_produtos.drop(columns=["categoria_norm", "categoria_final"], inplace=True)

df_produtos.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,ELETRONICOS
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,ELETRONICOS
4,Piloto Automático Furuno Storm,R$ 23669.01,5,ELETRONICOS
